# SNI-21 Paired Full-Frame Context Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ediprin/coffee-bean-detection/blob/agent/add-vadcp-pipeline/notebooks/SNI21_Local_Context_Control_Colab.ipynb)

Eksperimen **tanpa training dan tanpa test**. Tiga arm mempertahankan kanvas, bbox, posisi, ukuran, dan resolusi: gambar asli, repaste pada gambar asli, dan repaste pada background prosedural.

In [ ]:
from google.colab import drive
from pathlib import Path

if Path('/content/drive/MyDrive').is_dir():
    drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

import json, os, subprocess, sys
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
URL = 'https://github.com/ediprin/coffee-bean-detection.git'
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', URL, str(REPO)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
import coffee_detector, torch
assert torch.cuda.is_available(), 'Aktifkan runtime T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('IMPORT:', coffee_detector.__file__)
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO, check=True)

In [ ]:
DRIVE = Path('/content/drive/MyDrive')
SHORTCUTS = Path('/content/drive/.shortcut-targets-by-id')
direct_indexes = [
    DRIVE / 'Coffee_Bean_Detection/artifact_index.json',
    DRIVE / '02_RISET_DAN_PROYEK/Coffee_Bean_Detection/artifact_index.json',
    SHORTCUTS / '150ljaOEY25tOMecAmrctJX3VG2vSujso/Coffee_Bean_Detection/artifact_index.json',
]
index_matches = [path for path in direct_indexes if path.is_file()]
assert index_matches, 'artifact_index.json tidak ditemukan pada root proyek yang sudah dibakukan.'
PROJECT_INDEX = index_matches[0]
index = json.loads(PROJECT_INDEX.read_text(encoding='utf-8'))
assert index['project'] == 'coffee-bean-detection'
PROJECT_ROOT = PROJECT_INDEX.parent
artifacts = index['artifacts']
CHECKPOINT = PROJECT_ROOT / artifacts['a0_checkpoint']
A0_ARCHIVE = PROJECT_ROOT / artifacts['a0_real_archive_optional']
SOURCE_BENCHMARK_ROOT = PROJECT_ROOT / artifacts['density_benchmark_root']
DENSITY_EVALUATION_ROOT = PROJECT_ROOT / artifacts['density_evaluation_output']
assert CHECKPOINT.is_file(), CHECKPOINT
assert A0_ARCHIVE.is_file(), A0_ARCHIVE
assert (SOURCE_BENCHMARK_ROOT / 'val_object_library/object_library.json').is_file()
assert (SOURCE_BENCHMARK_ROOT / 'val_scene_profile.json').is_file()
assert (DENSITY_EVALUATION_ROOT / 'density_evaluation_summary.json').is_file()
R0_ROOT = Path('/content/sni21-fullscene-v1')
BENCHMARK_OUTPUT = Path('/content/sni21-fullframe-context-v1')
EVALUATION_OUTPUT = PROJECT_ROOT / 'experiments/sni21-fullframe-context-v1'
print('PROJECT   :', PROJECT_ROOT)
print('BENCHMARK :', BENCHMARK_OUTPUT)
print('EVALUATION:', EVALUATION_OUTPUT)

In [ ]:
from coffee_detector.archive_sni21_pilot import restore_real_a0_validation
restore_real_a0_validation(A0_ARCHIVE, R0_ROOT)
restore = json.loads((R0_ROOT / 'validation_restore.json').read_text(encoding='utf-8'))
assert restore['test_files_extracted'] == 0
assert restore['test_images_accessed'] is False
print(json.dumps(restore, indent=2))

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.run_sni21_local_context_control',
    '--checkpoint', str(CHECKPOINT),
    '--real-root', str(R0_ROOT),
    '--source-benchmark-root', str(SOURCE_BENCHMARK_ROOT),
    '--density-evaluation-root', str(DENSITY_EVALUATION_ROOT),
    '--benchmark-output-root', str(BENCHMARK_OUTPUT),
    '--evaluation-output-root', str(EVALUATION_OUTPUT),
    '--device', '0', '--imgsz', '640', '--batch-size', '8',
    '--confidence', '0.001', '--nms-iou', '0.7',
    '--diagnostic-iou', '0.5', '--max-det', '300',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
assert process.wait() == 0, 'Kontrol local-context gagal; baca traceback di atas.'

In [ ]:
import pandas as pd
from IPython.display import display
path = EVALUATION_OUTPUT / 'local_context_summary.json'
assert path.is_file(), path
summary = json.loads(path.read_text(encoding='utf-8'))
assert summary['training_executed'] is False
assert summary['test_images_accessed'] is False
table = pd.DataFrame(summary['rows'])
columns = ['map50_95', 'map50', 'precision', 'recall', 'macro_map50_95', 'bottom3_map50_95', 'worst_map50_95', 'proposal_recall_at_50', 'conditional_class_accuracy', 'proposal_miss_rate', 'localized_wrong_class_rate', 'saturation_rate']
display(table.style.format({column: '{:.2%}' for column in columns}))
print('\nSOURCE-SUBSET ATTRIBUTION')
print(json.dumps(summary['source_attribution'], indent=2))
print('\nCUTOUT ATTRIBUTION')
print(json.dumps(summary['cutout_attribution'], indent=2))
print('\nBACKGROUND ATTRIBUTION')
print(json.dumps(summary['background_attribution'], indent=2))
print('\nVALIDITY')
print(json.dumps(summary['validity'], indent=2))
print('\nSUMMARY:', path)
print('Kirim tabel, tiga attribution, dan validity ini. Jangan training model baru.')

In [ ]:
# Post-hoc paired analysis: membaca JSONL yang sudah ada; tidak ada inference/training.
analysis_command = [
    sys.executable, '-u', '-m', 'coffee_detector.analyze_sni21_fullframe_context',
    '--evaluation-root', str(EVALUATION_OUTPUT),
    '--iou-threshold', '0.5', '--iterations', '10000', '--seed', '42',
]
subprocess.run(analysis_command, cwd=REPO, check=True)
analysis_path = EVALUATION_OUTPUT / 'paired_context_analysis.json'
analysis = json.loads(analysis_path.read_text(encoding='utf-8'))
class_table = pd.DataFrame(analysis['class_table']).sort_values('delta_top1_accuracy')
display(class_table.style.format({
    column: '{:+.2%}' if column.startswith('delta_') else '{:.2%}'
    for column in class_table.columns
    if column.startswith(('fc1_', 'fc2_', 'delta_'))
}))
for key in ['micro_top1_accuracy', 'macro_top1_accuracy', 'macro_proposal_recall', 'transitions', 'paired_mcnemar_top1', 'stratified_bootstrap']:
    print(f'\n{key.upper()}')
    print(json.dumps(analysis[key], indent=2))
print('\nSUPPORTED:', analysis['paired_background_harm_supported'])
print('SUMMARY  :', analysis_path)
print('Kirim seluruh output sel ini. Tidak perlu menjalankan ulang sel inference jika JSONL sudah ada.')